In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


tickers = ["JNJ", "AAPL", "MSFT", "PG", "KO"]
start = "2017-01-01"
end = "2020-12-31"

stock_data = {}

/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
# Download market indext data for S&P 500 index ticker
sp500 = yf.download("^GSPC", start=start, end=end, auto_adjust=True, progress=False)
sp500.columns = sp500.columns.droplevel(1)
sp500_market_return = sp500['Close'].pct_change().to_frame(name='market_return')

,market_return
Date,
2017-01-03,NaN
2017-01-04,0.005722
2017-01-05,-0.000771
2017-01-06,0.003517
2017-01-09,-0.003549
...,...
2020-12-23,0.000746
2020-12-24,0.003537
2020-12-28,0.008723


In [3]:
# Download stock data for each ticker and store it in a dictionary
for ticker in tickers:
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    # Drop stock name level from columns
    df.columns = df.columns.droplevel(1)
    # Ensure datetime index and sort by date
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
    # Calculate stock returns variable
    df['stock_return'] = df['Close'].pct_change()
    # Target column is the stock return of the next day, so we shift the 'stock_return' column by -1
    df['target'] = df['stock_return'].shift(-1)
    # Add market indext return to the stock data
    df = df.merge(sp500_market_return, left_index=True, right_index=True, how='inner')

    # Add the stock data to the dictionary
    stock_data[ticker] = df

df = stock_data['AAPL']
df

,Close,High,Low,Open,Volume,stock_return,target,market_return
Date,,,,,,,,
2017-01-03,26.745853,26.787302,26.425778,26.665259,115127600,NaN,-0.001119,NaN
2017-01-04,26.715916,26.828749,26.653744,26.676770,84472400,-0.001119,0.005086,0.005722
2017-01-05,26.851782,26.909349,26.667565,26.692895,88774400,0.005086,0.011148,-0.000771
2017-01-06,27.151127,27.208694,26.819538,26.890921,127007600,0.011148,0.009160,0.003517
2017-01-09,27.399818,27.501138,27.158036,27.160337,134247600,0.009160,0.001009,-0.003549
...,...,...,...,...,...,...,...,...
2020-12-23,127.364151,128.793775,127.189086,128.531199,88223700,-0.006976,0.007712,0.000746
2020-12-24,128.346390,129.795483,127.500283,127.714243,54930100,0.007712,0.035766,0.003537
2020-12-28,132.936798,133.568945,129.844106,130.310937,124486200,0.035766,-0.013315,0.008723


### Preprocessing Functions

In [ ]:
def handle_outliers_winsorize(df, col, lower_q=0.01, upper_q=0.99):
    lower = df[col].quantile(lower_q)
    upper = df[col].quantile(upper_q)
    df[col] = df[col].clip(lower, upper)
    return df

# Handle missing values and align to calendar frequency
def set_calendar_frequency(df):
    """
    Align dataframe to a specific calendar frequency.

    Parameters:
    - df: pandas DataFrame (must have DatetimeIndex)

    Returns:
    - Resampled DataFrame
    """

    # Reindex to business days (adds missing trading days)
    full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='B') # Business day frequency
    df = df.reindex(full_index)
    # Fill missing values using forward fill (propagate last valid observation)
    df = df.ffill()

    return df

### Feature Engineering Functions

In [ ]:
def add_moving_average(df, n, price_col='Close'):
    """
    Calculate n-day Simple Moving Average (SMA) on a financial time series.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing price data
    n : int
        Window size (e.g., 10, 50, 200)
    price_col : str
        Column name to calculate SMA on (default: 'Close')

    Returns:
    --------
    df : pandas.DataFrame
        DataFrame with new SMA column added
    """
    
    df[f'SMA_{n}'] = df[price_col].rolling(window=n, min_periods=n).mean()
    return df

def add_volatility(df, n=20, price_col='Close', annualize=False):
    """
    Add an n-day rolling volatility feature based on past returns.

    This is a rolling-window feature engineering step:
    - returns describe day-to-day price changes
    - rolling std of returns estimates recent noise / uncertainty
    - if annualize=True, daily volatility is scaled by sqrt(252)

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing price data.
    n : int
        Rolling window size.
    price_col : str, default='Close'
        Column used to compute returns.
    annualize : bool, default=False
        If True, annualise daily volatility using sqrt(252).

    Returns
    -------
    pandas.DataFrame
        Copy of df with:
        - 'return'
        - f'volatility_{n}' (or annualised version)
    """

    # Rolling volatility = recent standard deviation of returns
    vol_col = f'volatility_{n}'
    df[vol_col] = df['stock_return'].rolling(window=n, min_periods=n).std()

    # Optional annualisation for finance applications
    if annualize:
        df[vol_col] = df[vol_col] * np.sqrt(252)

    return df

def add_momentum_rsi(df, n=14, price_col='Close', col_name=None):
    """
    Adds Relative Strength Index (RSI) feature to dataframe.

    Parameters:
    - df: pandas DataFrame
    - n: lookback window (default = 14, standard in finance)
    - price_col: column name for price (default = 'Close')
    - col_name: optional custom column name

    Returns:
    - df with RSI column added
    """
    
    if col_name is None:
        col_name = f'RSI_{n}'
    
    # Price changes (returns in price space, not percentage)
    delta = df[price_col].diff()

    # Gains and losses
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    gain = pd.Series(gain, index=df.index)
    loss = pd.Series(loss, index=df.index)

    # Rolling averages (Wilder's smoothing approximation using mean)
    avg_gain = gain.rolling(window=n, min_periods=n).mean()
    avg_loss = loss.rolling(window=n, min_periods=n).mean()

    # Avoid division by zero
    rs = avg_gain / (avg_loss + 1e-10)

    # RSI formula
    rsi = 100 - (100 / (1 + rs))

    df[col_name] = rsi

    return df

In [ ]:
# Handle missing values (calendar gaps) using forward fill
set_calendar_frequency(df)

# Feature Engineering: Add technical indicators as features
add_moving_average(df,10)
add_moving_average(df,50)
add_moving_average(df,200)
add_volatility(df)
add_momentum_rsi(df)

# Check for missing values after feature engineering. These can arise from rolling calculations (e.g., first 9 rows for SMA_10 will be NaN).
df.dropna(inplace=True) 

# Handle outliers
handle_outliers_winsorize(df,'stock_return')
handle_outliers_winsorize(df,'market_return')